# Purpose/risk through the service

The whole run as the pipeline will do it (spec `docs/specs/0003-purpose-risk-service.md`):
`PurposeRiskService` loads the prompt, writes the run's manifest, and joins the
loader, the controller and the store. It creates the controller, which creates the
model; this notebook never touches either.

Rows go to `data/dsld/runs/<ts>/raw/purpose_risk.jsonl`, one per label, failed ones
included. Progress and the ETA come from the controller's log.

In [ ]:
import logging

import pandas as pd

from engine.repositories.dsld import store
from engine.services.purpose_risk.service import PurposeRiskService

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s", force=True)

## Is the GPU free?

The model needs ~21 GB. Anything listed under compute apps is holding the card.

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_gpu_memory --format=csv

## A new run

64 labels from the frozen pilot passages, in file order. Leaving the `with` block
closes the controller, which stops the server and frees the card -- also on an
error or an interrupt.

In [ ]:
with PurposeRiskService() as svc:
    svc.run(limit=64)

## The rows

The newest run directory, one row per label.

In [ ]:
run = sorted(store.RUNS_ROOT.glob("*T*Z*"))[-1]
print(run)
print(store.read_manifest(run))

df = pd.DataFrame(store.read_rows(run))
print(df.status.value_counts().to_string())
df[["dsld_id", "status", "attempts", "failed", "purpose", "risk", "latency_ms"]]

## One failed label in full

The raw answer is kept on every row, so a failure can be read without the GPU.

In [ ]:
failed = df[df.status != "ok"]
failed.iloc[0].to_dict() if len(failed) else "no failed labels"

## Resume

Interrupt the run above (stop the kernel's cell) and run this: it reads the selection
from the manifest and does only the labels the run does not hold yet. A changed
prompt or sampling setting is refused -- start a new run instead.

In [ ]:
with PurposeRiskService() as svc:
    svc.run(run=run)